In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
orders = pd.read_excel("PaintShop - September 2026.xlsx", sheet_name=0)
machines = pd.read_excel("PaintShop - September 2026.xlsx", sheet_name=1)
setups = pd.read_excel("PaintShop - September 2026.xlsx", sheet_name=2)

orders["Start"] = [0.0 for i in range(len(orders))] #Voegt een column toe voor de start tijd

orders["End"] = [0.0 for i in range(len(orders))] #Voegt een column toe voor de eind tijd

orders["Penalty_order"] = [0.0 for i in range(len(orders))] #Voegt een column toe voor de opgelopen penalty van een order

orders["Done"] = [0 for i in range(len(orders))] #Voegt een column toe om aan te geven dat de order klaar is

orders["Machine"] = ["None" for i in range(len(orders))] #Voegt een column toe die aangeft welke machine het heeft gedaan

orders = orders.sort_values(by = ["Colour", "Surface"]) #Sorteert de dataframe op kleur en oppervlakte

machines["Colour"] = ["None" for i in range(len(machines))] #Voegt een column toe die aangeeft welke kleur de machine is

machines["Total_time"] = [0.0 for i in range(len(machines))] #Voegt een column toe voor de totale tijd dat de machine heeft gewerkt

machines = machines.sort_values(by = ["Speed"]).reset_index(drop=True) #Sorteert de machines op snelheid

for i in setups["To colour"].unique():
    setups.loc[len(setups)] = ["None", i, 0] #Voegt de setups toe van geen kleur naar een kleur
    setups.loc[len(setups)] = [i, i, 0] #Voegt de setups toe van dezelfde kleur naar dezelfde kleur

Greedy constructive heuristic
1. Pak de traagste machine die niet bezig is en geef die de order die die als eerst klaar heeft neem de tijd van een potentiële kleur wissel hierin mee.
2. Herhaal stap 1 tot alle machines bezig zijn en er nog orders open staan

In [9]:
while orders.Done.sum() < len(orders):
    for machine in machines.index:
        not_done = orders[orders.Done == 0] #Maakt een lijst van alle orders dat nog niet klaar zijn
        if len(not_done) == 0:
            break #Breekt de while loop als alle orders zijn gedaan
        choices = pd.DataFrame({"Order": ["None"], #Maakt/reset de DataFrame voor de keuzes waar een machine uit moet kieze
                "Surface": [0],
                "Colour": ["None"]})
        for i in range(len(not_done.Colour.unique())):
            Colour = not_done.Colour.unique()[i]
            choices.loc[i] = list(not_done[["Order","Surface", "Colour"]][(not_done.Colour == Colour)].reset_index(drop=True).loc[0]) #Voegt een keuze toe aan de dataframe

        choices["Time"] = [choices.Surface.iloc[n]/machines.Speed.loc[machine] +
                    (setups["Setup time"][(setups["From colour"] == machines.Colour.loc[machine]) &
                                            (setups["To colour"] == choices.Colour.loc[n])].iloc[0]) for n in range(len(choices))] #Voegt een extra column toe die aangeeft hoelang een keuze gaat duren

        choice = choices.sort_values(by = "Time").reset_index(drop=True).iloc[0] #Kiest de snelste keuze

        ind = orders.index[orders.Order == choice.Order][0] #index van de order in de orders dataframe
        orders.at[ind, "Start"] = machines.Total_time.loc[machine] + (setups["Setup time"][(setups["From colour"] == machines.Colour.loc[machine]) &
                                            (setups["To colour"] == choice.Colour)].iloc[0]) #Vult de start tijd van de order in
        orders.at[ind, "End"] = machines.Total_time.loc[machine] + choice.Time #Vult de eind tijd van de order in
        if machines.Total_time.loc[machine] + choice.Time > orders.Deadline.iloc[ind]:
            orders.at[ind, "Penalty_order"] = (machines.Total_time.loc[machine] + choice.Time - orders.Deadline.iloc[ind])*orders.Penalty.iloc[ind] #Vult de penalty score van de order in
        orders.at[ind, "Done"] = 1 #Geeft aan dat de order klaar is
        orders.at[ind, "Machine"] = machines.Machine.loc[machine] #Geeft aan welke machine de order heeft gedaan

        machines.at[machine, "Total_time"] = machines.Total_time.loc[machine] + choice.Time #Geeft de totale werktijd van de machines
        machines.at[machine, "Colour"] = choice.Colour #Geeft de kleur van de machine
        machines = machines.sort_values(by = ["Total_time", "Speed"], ascending = True) #Sorteert de dataframe op totale werktijd van de machines
print(f'Total penalty score is {orders.Penalty_order.sum()}')
orders.sort_values(by = ["Machine", "Start"])

Total penalty score is 3141.583333333333


,Order,Surface,Colour,Deadline,Penalty,Start,End,Penalty_order,Done,Machine
26,ORD27,120,Blue,78,8,0.000000,6.000000,0.000000,1,M1
5,ORD06,160,Blue,31,11,6.000000,14.000000,0.000000,1,M1
15,ORD16,160,Blue,63,8,14.000000,22.000000,0.000000,1,M1
3,ORD04,170,Blue,15,16,22.000000,30.500000,248.000000,1,M1
22,ORD23,180,Blue,104,9,30.500000,39.500000,0.000000,1,M1
18,ORD19,200,Blue,74,15,39.500000,49.500000,189.000000,1,M1
23,ORD24,190,Red,113,12,57.500000,67.000000,468.000000,1,M1
11,ORD12,220,Red,18,6,67.000000,78.000000,602.000000,1,M1
10,ORD11,230,Yellow,20,9,86.000000,97.500000,115.000000,1,M1
9,ORD10,120,Green,43,10,0.000000,5.000000,0.000000,1,M2


In [277]:
orders

,Order,Surface,Colour,Deadline,Penalty,Start,End,Done
26,ORD27,120,Blue,78,8,0.000000,6.000000,1
5,ORD06,160,Blue,31,11,6.000000,14.000000,1
15,ORD16,160,Blue,63,8,14.000000,22.000000,1
3,ORD04,170,Blue,15,16,22.000000,30.500000,1
2,ORD03,180,Blue,14,8,26.250000,36.750000,1
22,ORD23,180,Blue,104,9,30.500000,39.500000,1
8,ORD09,190,Blue,32,7,36.750000,44.666667,1
18,ORD19,200,Blue,74,15,39.500000,49.500000,1
12,ORD13,110,Green,49,11,0.000000,6.111111,1
9,ORD10,120,Green,43,10,0.000000,5.000000,1
